# EDL Full Experiment (auto-parallel, per-(dataset,model) hyperparams)

10-fold cross-validation comparison of 8 LDL models across the datasets
listed in `DATASETS`, parallelised via `loky`. The pool config is
auto-detected by `edl_workers.auto_pool_config()`:

- **GPU mode** if ≥ 2 CUDA GPUs are visible to `nvidia-smi` — one worker
  pinned per GPU.
- **CPU mode** otherwise — a small number of fat CPU workers (≈
  `cpu_count // 4`).

The training loop lives in `edl_workers.py` next to this notebook so loky
subprocesses can `import edl_workers` cleanly. TensorFlow is imported
lazily inside each worker after `CUDA_VISIBLE_DEVICES` is pinned.

For each (model, dataset) pair we record six distributional metrics
(`chebyshev`, `clark`, `canberra`, `kl_divergence`, `cosine`,
`intersection`) across 10 folds, plus per-sample uncertainty for the
evidential / SNEFY models.

Hyperparameters are configured per-(dataset, model) in the **Hyperparams**
section below. Anything you don't override falls back to `DEFAULT_HP`.


In [1]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import sys
import multiprocessing as mp
from collections import defaultdict
from concurrent.futures import as_completed
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from loky import get_reusable_executor

# Make edl_workers importable whether the kernel started in `demo/` or in the
# project root.
_demo_dir = Path.cwd() if (Path.cwd() / 'edl_workers.py').exists() else Path.cwd() / 'demo'
if str(_demo_dir) not in sys.path:
    sys.path.insert(0, str(_demo_dir))

from pyldl.utils import load_dataset
from edl_workers import (
    init_worker, run_one_fold, auto_pool_config,
    MODEL_NAMES, METRICS, DEFAULT_HP,
)


E0000 00:00:1777403245.877517 2324398 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777403246.346770 2324398 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777403248.345587 2324398 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777403248.345630 2324398 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777403248.345632 2324398 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777403248.345633 2324398 computation_placer.cc:177] computation placer already registered. Please check linka

ImportError: cannot import name 'DEFAULT_HP' from 'edl_workers' (/project/6004619/dcs01/PyLDL/demo/edl_workers.py)

## Datasets and CV settings


In [ ]:
DATASETS     = ['SJAFFE', 'SBU_3DFE', 'Movie']
N_SPLITS     = 10
RANDOM_STATE = 0


## Hyperparams

`DEFAULT_HP` (defined in `edl_workers.py`) supplies the fallback for any
key that isn't overridden:

```
n_hidden       64       # dense width for the encoder/MLP
learning_rate  1e-3     # AdamW; set to None to use each model's own optimizer
weight_decay   1e-4     # AdamW
dropout_rate   0.0      # inserted between hidden and output if > 0
epochs         2500     # upper bound; early stopping cuts it short
batch_size     None     # None → each model's own default
val_fraction   0.1      # of the training fold; held out as inner-val for early-stop
patience       50       # LDLEarlyStopping; set None to disable
minimum        100      # LDLEarlyStopping; minimum epochs before early-stop can fire
max_iterations 50       # SA_BFGS only
```

Override per (dataset, model) in `PER_DATASET_HP` below. Anything you
don't list uses `DEFAULT_HP` as-is.


In [ ]:
PER_DATASET_HP = {
    'SJAFFE': {
        'AA_BP':                    {'patience': 100, 'minimum': 100, 'n_hidden': 64},
        'Duo_LDL':                  {'patience': 100, 'minimum': 100, 'n_hidden': 64},
        'EDL_LDL (loglikelihood)':  {'patience': 100, 'minimum': 100, 'n_hidden': 64},
        'EDL_LDL (bayes_mse)':      {'patience': 100, 'minimum': 100, 'n_hidden': 64},
        'BEDL_LDL (loglikelihood)': {'patience': 100, 'minimum': 100, 'n_hidden': 64},
        'BEDL_LDL (bayes_mse)':     {'patience': 100, 'minimum': 100, 'n_hidden': 64},
        'SNEFY_LDL':                {'patience': 100, 'minimum': 100, 'n_hidden': 64},
        'SA_BFGS':                  {},
    },
    'SBU_3DFE': {
        'AA_BP':                    {'patience': 20, 'minimum': 20, 'n_hidden': 32, 'weight_decay': 1e-4, 'batch_size': 256},
        'Duo_LDL':                  {'patience': 20, 'minimum': 20, 'n_hidden': 32, 'weight_decay': 1e-4, 'batch_size': 256},
        'EDL_LDL (loglikelihood)':  {'patience': 20, 'minimum': 20, 'n_hidden': 32, 'weight_decay': 1e-4, 'batch_size': 256},
        'EDL_LDL (bayes_mse)':      {'patience': 20, 'minimum': 20, 'n_hidden': 32, 'weight_decay': 1e-4, 'batch_size': 256},
        'BEDL_LDL (loglikelihood)': {'patience': 20, 'minimum': 20, 'n_hidden': 32, 'weight_decay': 1e-4, 'batch_size': 256},
        'BEDL_LDL (bayes_mse)':     {'patience': 20, 'minimum': 20, 'n_hidden': 32, 'weight_decay': 1e-4, 'batch_size': 256},
        'SNEFY_LDL':                {'patience': 20, 'minimum': 20, 'n_hidden': 32, 'weight_decay': 1e-4, 'batch_size': 256},
        'SA_BFGS':                  {},
    },
    'Natural_Scene': {
        'AA_BP':                    {},
        'Duo_LDL':                  {},
        'EDL_LDL (loglikelihood)':  {},
        'EDL_LDL (bayes_mse)':      {},
        'BEDL_LDL (loglikelihood)': {},
        'BEDL_LDL (bayes_mse)':     {},
        'SNEFY_LDL':                {},
        'SA_BFGS':                  {},
    },
    'Movie': {
        'AA_BP':                    {'patience': 20, 'minimum': 20, 'n_hidden': 4, 'weight_decay': 1e-3, 'batch_size': 512},
        'Duo_LDL':                  {'patience': 20, 'minimum': 20, 'n_hidden': 4, 'weight_decay': 1e-3, 'batch_size': 512},
        'EDL_LDL (loglikelihood)':  {'patience': 20, 'minimum': 20, 'n_hidden': 4, 'weight_decay': 1e-3, 'batch_size': 512},
        'EDL_LDL (bayes_mse)':      {'patience': 20, 'minimum': 20, 'n_hidden': 4, 'weight_decay': 1e-3, 'batch_size': 512},
        'BEDL_LDL (loglikelihood)': {'patience': 20, 'minimum': 20, 'n_hidden': 4, 'weight_decay': 1e-3, 'batch_size': 512},
        'BEDL_LDL (bayes_mse)':     {'patience': 20, 'minimum': 20, 'n_hidden': 4, 'weight_decay': 1e-3, 'batch_size': 512},
        'SNEFY_LDL':                {'patience': 20, 'minimum': 20, 'n_hidden': 4, 'weight_decay': 1e-3, 'batch_size': 512},
        'SA_BFGS':                  {},
    },
}


def resolve_hp(dataset, model_name):
    """Merge DEFAULT_HP with the per-(dataset, model) overrides."""
    hp = dict(DEFAULT_HP)
    hp.update(PER_DATASET_HP.get(dataset, {}).get(model_name, {}))
    return hp


## Pool configuration (auto-detected)


In [ ]:
# Set POOL_CFG to a dict to override; leave None for auto-detect.
POOL_CFG = None
if POOL_CFG is None:
    POOL_CFG = auto_pool_config()

GPU_IDS   = POOL_CFG['gpu_ids']
N_WORKERS = POOL_CFG['n_workers']
INTRA     = POOL_CFG['intra_op_threads']
INTER     = POOL_CFG['inter_op_threads']
MODE      = POOL_CFG['mode']

print(f'mode      : {MODE}')
print(f'workers   : {N_WORKERS}')
print(f'gpu_ids   : {GPU_IDS if GPU_IDS else "(CPU only)"}')
print(f'tf threads: intra={INTRA}, inter={INTER}')


## Build the worker pool


In [ ]:
mgr = mp.Manager()
gpu_queue = mgr.Queue()

slots = list(GPU_IDS) if GPU_IDS else [None] * N_WORKERS
assert len(slots) == N_WORKERS, 'one queue slot per worker'
for g in slots:
    gpu_queue.put(g)

executor = get_reusable_executor(
    max_workers=N_WORKERS,
    initializer=init_worker,
    initargs=(gpu_queue, INTRA, INTER),
    reuse=False,
)
print(f'pool ready: {N_WORKERS} workers, slots={slots}')


## Sanity check: what do the workers actually see?

`gpu_devices` should match what you expect for the mode:

- **GPU mode**: each worker has a different `CUDA_VISIBLE_DEVICES` and
  reports a single GPU device.
- **CPU mode**: every worker reports `CUDA_VISIBLE_DEVICES = '-1'` and
  an empty `gpu_devices` list.


In [ ]:
def _check():
    import os, tensorflow as tf
    return {
        'pid': os.getpid(),
        'CUDA_VISIBLE_DEVICES': os.environ.get('CUDA_VISIBLE_DEVICES'),
        'gpu_devices': [d.name for d in tf.config.list_physical_devices('GPU')],
        'intra_threads': tf.config.threading.get_intra_op_parallelism_threads(),
        'inter_threads': tf.config.threading.get_inter_op_parallelism_threads(),
    }

checks = [executor.submit(_check) for _ in range(N_WORKERS * 4)]
df = pd.DataFrame([c.result() for c in checks]).drop_duplicates(subset='pid').reset_index(drop=True)
print(f'{len(df)} unique workers (expected {N_WORKERS})')
df


## Build the job list

Each job carries the resolved `hp` dict for its (dataset, model)
combination, so workers don't need to know about `PER_DATASET_HP`.


In [ ]:
jobs = []
for dataset_name in DATASETS:
    X, D = load_dataset(dataset_name, dir='dataset')
    print(f'{dataset_name}: {X.shape[0]} samples, {X.shape[1]} features, {D.shape[1]} labels')
    kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    for fold_idx, (train_idx, test_idx) in enumerate(kf.split(X), start=1):
        Xtr, Xte = X[train_idx], X[test_idx]
        Dtr, Dte = D[train_idx], D[test_idx]
        for model_name in MODEL_NAMES:
            hp = resolve_hp(dataset_name, model_name)
            jobs.append((dataset_name, model_name, fold_idx, Xtr, Dtr, Xte, Dte, hp))

total = len(jobs)
print(f'queued {total} jobs ({len(DATASETS)} datasets × {N_SPLITS} folds × {len(MODEL_NAMES)} models)')


### Inspect the resolved hyperparams

Confirm the config you'll actually run with — defaults plus your
per-(dataset, model) overrides.


In [ ]:
hp_rows = []
for ds in DATASETS:
    for m in MODEL_NAMES:
        hp = resolve_hp(ds, m)
        hp_rows.append({'dataset': ds, 'model': m, **hp})
hp_table = pd.DataFrame(hp_rows).set_index(['dataset', 'model'])
hp_table


## Submit + collect

Submission is non-blocking; results stream back via `as_completed`. Per-fold
failures are caught and logged but don't stop the run.


In [ ]:
futures = {
    executor.submit(run_one_fold, ds, m, fi, Xtr, Dtr, Xte, Dte, hp): (ds, m, fi)
    for (ds, m, fi, Xtr, Dtr, Xte, Dte, hp) in jobs
}

raw_results = []
failures = []
for i, fut in enumerate(as_completed(futures), start=1):
    ds, m, fi = futures[fut]
    try:
        raw_results.append(fut.result())
        status = 'ok'
    except Exception as e:
        failures.append((ds, m, fi, repr(e)))
        status = f'FAILED ({type(e).__name__}: {e})'
    print(f'[{i:4d}/{total}] {ds:12s} | fold {fi:2d} | {m:30s} {status}')

print(f'\ndone: {len(raw_results)} ok, {len(failures)} failed')


### Worker-state diagnostic

If the submit cell hangs, run this in a separate cell to see if any
workers have died. Healthy state = all workers `sleeping`/`running` with
non-trivial `rss`. Dead PIDs in the `dead` list = OOM-killed or crashed,
in which case rebuild the pool and resubmit only `missing` jobs.


In [ ]:
import psutil

worker_pids = list(executor._processes.keys())
alive, dead = [], []
for pid in worker_pids:
    try:
        p = psutil.Process(pid)
        if p.is_running() and p.status() != psutil.STATUS_ZOMBIE:
            alive.append((pid, p.status(), p.memory_info().rss / 1e9, p.cpu_percent(interval=0.5)))
        else:
            dead.append(pid)
    except psutil.NoSuchProcess:
        dead.append(pid)

print(f'alive workers ({len(alive)}/{N_WORKERS}):')
for pid, status, rss_gb, cpu in alive:
    print(f'  pid={pid}  status={status}  rss={rss_gb:.1f}GB  cpu={cpu:.0f}%')
print(f'dead workers: {dead}')
print(f'\nresults so far: {len(raw_results)} / {total}')
print(f'completed futures: {sum(f.done() for f in futures)}')
print(f'pending futures:   {sum(not f.done() for f in futures)}')


## Bucket results into per-model DataFrames


In [ ]:
buckets = defaultdict(list)
for r in raw_results:
    buckets[(r['dataset'], r['model'])].append(r['scores'])

per_model_results = {key: pd.DataFrame(rows) for key, rows in buckets.items()}
print(f'{len(per_model_results)} (dataset, model) combinations have results')


## Per-model fold tables


In [ ]:
if per_model_results:
    first_key = next(iter(per_model_results))
    print(f'showing: {first_key}')
    display(per_model_results[first_key])
else:
    print('no results yet — run the submit cell first')


## Combined summary — mean ± std across folds


In [ ]:
def summarize(df):
    out = {}
    for col in df.columns:
        out[f'{col}_mean'] = df[col].mean()
        out[f'{col}_std']  = df[col].std()
    return out


summary_rows = []
for (dataset_name, model_name), df in per_model_results.items():
    if df.empty:
        continue
    row = {'dataset': dataset_name, 'model': model_name, **summarize(df)}
    summary_rows.append(row)

summary = pd.DataFrame(summary_rows).set_index(['dataset', 'model'])
summary


### Compact view: `mean ± std` per metric


In [ ]:
def fmt(mean, std):
    if pd.isna(mean):
        return ''
    return f'{mean:.4f} ± {std:.4f}'


compact_rows = []
for (dataset_name, model_name), df in per_model_results.items():
    if df.empty:
        continue
    row = {'dataset': dataset_name, 'model': model_name}
    for col in df.columns:
        row[col] = fmt(df[col].mean(), df[col].std())
    compact_rows.append(row)

compact = pd.DataFrame(compact_rows).set_index(['dataset', 'model'])
compact


## Uncertainty results (EDL_LDL, BEDL_LDL, SNEFY_LDL only)

- `mean_uncertainty` — average per-sample uncertainty on test (model-specific
  scale; lower = more confident).
- `uncertainty_calibration` — Spearman ρ between per-sample uncertainty and
  per-sample KL divergence error. Higher = uncertainty better predicts error.


In [ ]:
uncertainty_models = {
    'EDL_LDL (loglikelihood)', 'EDL_LDL (bayes_mse)',
    'BEDL_LDL (loglikelihood)', 'BEDL_LDL (bayes_mse)',
    'SNEFY_LDL',
}

uncertainty_rows = []
for (dataset_name, model_name), df in per_model_results.items():
    if model_name not in uncertainty_models or df.empty:
        continue
    if 'mean_uncertainty' not in df.columns:
        continue
    uncertainty_rows.append({
        'dataset': dataset_name,
        'model': model_name,
        'mean_uncertainty':        fmt(df['mean_uncertainty'].mean(),        df['mean_uncertainty'].std()),
        'uncertainty_calibration': fmt(df['uncertainty_calibration'].mean(), df['uncertainty_calibration'].std()),
    })

uncertainty_summary = pd.DataFrame(uncertainty_rows).set_index(['dataset', 'model'])
uncertainty_summary


## Shut down the pool


In [ ]:
executor.shutdown(wait=True, kill_workers=True)
